# Five-year Coinbase hourly-candle backfill

This notebook downloads five years of completed hourly candles for the four-coin strategy universe: BTC, ETH, SOL, and DOGE. It shares the same SQLite database as `crypto-candles`, is safe to interrupt and rerun, and never places trades. XRP is intentionally excluded.

In [1]:
import logging
from pathlib import Path

import pandas as pd

from crypto_trader.data.coinbase_candles import (
    backfill_historical_candles,
    open_candle_database,
)

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
database_path = project_root / "data" / "database" / "trading.db"
products = ["BTC-USD", "ETH-USD", "SOL-USD", "DOGE-USD"]
backfill_years = 5
granularity = "ONE_HOUR"
interval_seconds = 60 * 60
request_delay_seconds = 0.1
logging.getLogger("crypto_trader.data.coinbase_candles").setLevel(logging.INFO)
connection = open_candle_database(database_path)

## Backfill

Run this cell to download the history. Coinbase limits each response to 350 candles, so the function walks through roughly 126 pages per product. It pauses briefly between requests, retries temporary failures, reports progress every ten pages, and skips complete pages when rerun. Coinbase occasionally omits a historical bucket; those omissions appear as warnings, but the backfill continues and does not invent replacement prices.

In [2]:
processed_rows = backfill_historical_candles(
    connection,
    products=products,
    years=backfill_years,
    granularity=granularity,
    request_delay=request_delay_seconds,
)
print(f"Processed {processed_rows:,} candle rows.")

Coinbase omitted 3 ONE_HOUR candle(s) for BTC-USD; first missing open_time_ms=1677952800000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 5 ONE_HOUR candle(s) for BTC-USD; first missing open_time_ms=1761408000000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 5 ONE_HOUR candle(s) for BTC-USD; first missing open_time_ms=1778205600000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 3 ONE_HOUR candle(s) for ETH-USD; first missing open_time_ms=1677952800000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 5 ONE_HOUR candle(s) for ETH-USD; first missing open_time_ms=1761408000000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 5 ONE_HOUR candle(s) for ETH-USD; first missing open_time_ms=1778205600000. Keeping the gap rather than inventing OHLCV data.
Coinbase omitted 5 ONE_HOUR candle(s) for SOL-USD; first missing open_time_ms=1677949200000. Keeping the gap rather than inventing OHLCV data.

Processed 175,244 candle rows.


## Inspect coverage and schema

These queries show the time coverage for each product, any missing source hours, recent prices, and the SQLite schema.

In [3]:
observed_times = pd.read_sql_query(
    """
    SELECT symbol, open_time_ms
    FROM candles
    WHERE exchange = 'coinbase' AND interval_seconds = ?
    ORDER BY symbol, open_time_ms
    """,
    connection,
    params=(interval_seconds,),
)

gap_rows = []
for symbol, rows in observed_times.groupby("symbol"):
    observed = pd.Index(rows["open_time_ms"])
    interval_ms = interval_seconds * 1_000
    expected = pd.RangeIndex(observed.min(), observed.max() + interval_ms, interval_ms)
    for open_time_ms in expected.difference(observed):
        gap_rows.append({"symbol": symbol, "open_time_ms": open_time_ms})

source_gaps = pd.DataFrame(gap_rows, columns=["symbol", "open_time_ms"])
if not source_gaps.empty:
    source_gaps["open_time_utc"] = pd.to_datetime(
        source_gaps["open_time_ms"], unit="ms", utc=True
    )
source_gaps


,symbol,open_time_ms,open_time_utc
0,BTC-USD,1677952800000,2023-03-04 18:00:00+00:00
1,BTC-USD,1677956400000,2023-03-04 19:00:00+00:00
2,BTC-USD,1677960000000,2023-03-04 20:00:00+00:00
3,BTC-USD,1761408000000,2025-10-25 16:00:00+00:00
4,BTC-USD,1761411600000,2025-10-25 17:00:00+00:00
5,BTC-USD,1761415200000,2025-10-25 18:00:00+00:00
6,BTC-USD,1761418800000,2025-10-25 19:00:00+00:00
7,BTC-USD,1761422400000,2025-10-25 20:00:00+00:00
8,BTC-USD,1778205600000,2026-05-08 02:00:00+00:00
9,BTC-USD,1778209200000,2026-05-08 03:00:00+00:00


In [4]:
coverage = pd.read_sql_query(
    """
    SELECT exchange, symbol, interval_seconds, COUNT(*) AS rows,
           MIN(open_time_ms) AS first_open_time_ms,
           MAX(open_time_ms) AS last_open_time_ms
    FROM candles
    WHERE exchange = 'coinbase' AND interval_seconds = ?
    GROUP BY exchange, symbol, interval_seconds
    ORDER BY symbol
    """,
    connection,
    params=(interval_seconds,),
)
coverage

,exchange,symbol,interval_seconds,rows,first_open_time_ms,last_open_time_ms
0,coinbase,BTC-USD,3600,43812,1633806000000,1791572400000
1,coinbase,DOGE-USD,3600,43810,1633806000000,1791572400000
2,coinbase,ETH-USD,3600,43812,1633806000000,1791572400000
3,coinbase,SOL-USD,3600,43810,1633806000000,1791572400000


In [5]:
latest = pd.read_sql_query(
    """
    SELECT symbol, datetime(open_time_ms / 1000, 'unixepoch') AS open_time_utc,
           open, high, low, close, volume
    FROM candles
    WHERE exchange = 'coinbase' AND interval_seconds = ?
    ORDER BY open_time_ms DESC, symbol
    LIMIT 25
    """,
    connection,
    params=(interval_seconds,),
)
latest

,symbol,open_time_utc,open,high,low,close,volume
0,BTC-USD,2026-10-09 19:00:00,82369.12000,82495.22000,82198.00000,82252.59000,2.154158e+02
1,DOGE-USD,2026-10-09 19:00:00,0.08456,0.08473,0.08410,0.08430,2.786500e+06
2,ETH-USD,2026-10-09 19:00:00,2480.94000,2485.51000,2472.09000,2474.89000,3.142447e+03
3,SOL-USD,2026-10-09 19:00:00,109.40000,109.63000,108.31000,108.38000,2.781414e+04
4,BTC-USD,2026-10-09 18:00:00,82643.61000,82664.81000,82233.09000,82369.11000,1.577893e+02
5,DOGE-USD,2026-10-09 18:00:00,0.08489,0.08495,0.08430,0.08456,4.859092e+06
6,ETH-USD,2026-10-09 18:00:00,2485.97000,2489.08000,2475.91000,2481.13000,5.691357e+03
7,SOL-USD,2026-10-09 18:00:00,109.83000,109.92000,109.06000,109.38000,2.347598e+04
8,BTC-USD,2026-10-09 17:00:00,82668.39000,82715.90000,82428.28000,82643.61000,1.721534e+02
9,DOGE-USD,2026-10-09 17:00:00,0.08476,0.08494,0.08437,0.08488,5.634223e+06


In [6]:
schema = pd.read_sql_query(
    "SELECT type, name, sql FROM sqlite_master WHERE sql IS NOT NULL ORDER BY type, name",
    connection,
)
schema

,type,name,sql
0,index,idx_candles_interval_open_time,CREATE INDEX idx_candles_interval_open_time\nO...
1,table,candles,CREATE TABLE candles (\n -- Internal row id...


In [7]:
connection.close()